In [ ]:
import pandas as pd

print("1. Загружаем исходные таблицы...")
try:
    sightings = pd.read_csv('../data/sightings.csv')
    crimes = pd.read_csv('../data/crime_incidents.csv')
    locations = pd.read_csv('../data/locations.csv')
    weather = pd.read_csv('../data/weather.csv')
except FileNotFoundError:
    sightings = pd.read_csv('data/sightings.csv')
    crimes = pd.read_csv('data/crime_incidents.csv')
    locations = pd.read_csv('data/locations.csv')
    weather = pd.read_csv('data/weather.csv')

print("2. Подготавливаем время для объединения с погодой...")
sightings['timestamp'] = pd.to_datetime(sightings['timestamp'])
sightings['date'] = sightings['timestamp'].dt.date
sightings['hour'] = sightings['timestamp'].dt.hour

print("3. Объединяем все таблицы в единый датасет...")
merged_df = sightings.merge(crimes, left_on='nearest_crime_id', right_on='crime_id', how='left', suffixes=('', '_crime'))
merged_df = merged_df.merge(locations, on='district', how='left', suffixes=('', '_loc'))
merged_df = merged_df.merge(weather, on=['date', 'hour'], how='left', suffixes=('', '_weather'))

print("4. Сохраняем результат в full_dataset.csv...")
try:
    merged_df.to_csv('../data/full_dataset.csv', index=False)
except:
    merged_df.to_csv('data/full_dataset.csv', index=False)

print(f"Успешно! Итоговый размер датасета: {merged_df.shape[0]} строк, {merged_df.shape[1]} колонок.")

# 1. Постановка задачи (Business Understanding)

### 1.1. Описание данных
Представленный датасет представляет собой систему мониторинга городской активности и происшествий (Masked Hero Tracker). Данные содержат информацию о наблюдениях, инцидентах, погодных условиях и локациях.

### 1.2. Условный заказчик
Заказчиком выступает аналитический отдел городской администрации, заинтересованный в оценке безопасности районов и анализе обстановки.

### 1.3. Задачи интеллектуального анализа данных
1. Описательная аналитика: поиск закономерностей между погодными условиями и частотой инцидентов.
2. Поиск аномалий: выявление резких всплесков активности в районах города.

# 2. Паспорт датасета (Data Understanding)
Ниже выполнены загрузка объединенных данных, проверка их размеров и формирование детального паспорта признаков.

In [ ]:
import pandas as pd

# 1. Загрузка данных в pandas DataFrame
df = pd.read_csv('../data/full_dataset.csv')

# 2. Определение размера датасета
rows_count, cols_count = df.shape
print(f'Количество строк (объектов): {rows_count}')
print(f'Количество столбцов (признаков): {cols_count}\n')

# 3. Формирование паспорта признаков (типы, заполненность, пропуски)
passport_df = pd.DataFrame({
    'Название признака': df.columns,
    'Тип данных (pandas)': df.dtypes.values,
    'Количество непустых строк': df.notnull().sum().values,
    'Доля пропусков (%)': (df.isnull().mean().values * 100).round(1)
})

display(passport_df)

### Описание основных групп признаков датасета:
* **Идентификаторы и ключи:** `sighting_id`, `crime_id`, `district` — служебные поля для связи таблиц и уникальной идентификации записей.
* **Временные метки:** `timestamp`, `date`, `hour` — фиксируют время и дату происшествия/наблюдения.
* **Географические координаты:** `latitude`, `longitude`, `borough`, `h3_cell` — пространственные характеристики событий.
* **Поведенческие и описательные признаки:** `report_type`, `witness_count`, `tracker_confidence`, признак наличия медиафайлов (`photo_evidence`, `video_evidence`).
* **Метеорологические данные:** `temperature_c`, `precipitation_mm`, `weather_condition`, `visibility_m` — параметры погоды в момент события.
* **Целевые переменные:** `verification_status` (мультикласс) и `is_verified` (бинарный таргет).